# Finite spatial vacuum root products: fail-closed Colab check

This intermediate-brick notebook checks one immutable WIP SHA. It validates the exact rational gate before compiling. It proves neither the Stieltjes/log-mixture identity nor either spectral-sector estimate.

In [ ]:
from pathlib import Path
import datetime, hashlib, json, os, platform, re, shutil, subprocess, sys, tempfile

REPO_URL = 'https://github.com/lluiseriksson/THE-ERIKSSON-PROGRAMME.git'
EXPECTED_SHA = '7e9d05aa895177076060b90799a852e767dca172'
EXPECTED_TOOLCHAIN = 'leanprover/lean4:v4.29.0-rc6'
EXPECTED_LEAN_COMMIT = '00659f8e6071d7e46131ed643bf8003b99b044e9'
EXPECTED_MATHLIB = '07642720480157414db592fa85b626dafb71355b'
EXPECTED_GATE_SHA256 = '14edcce2d80ecf3c83052700f97e8d5cae1aa8a54a93f4559d3e8ec67aee4cb6'
EXPECTED_GATE_OUTPUT_SHA256 = '1fa0c1f98608446606e44aad0d5a4e994272e7ef3d8ce89a5b4359d68d548de6'
EXPECTED_CORE_JOBS = 8466
RUN_ROOT = Path(tempfile.mkdtemp(prefix='spatial-root-products-'))
REPO = RUN_ROOT / 'repo'
ARTIFACTS = RUN_ROOT / 'artifacts'
ARTIFACTS.mkdir()
TRANSCRIPT = ARTIFACTS / 'transcript.txt'

def log(text):
    text = str(text)
    print(text)
    with TRANSCRIPT.open('a', encoding='utf-8', newline='\n') as stream:
        stream.write(text + '\n')

def run(cmd, cwd=None, env=None):
    shown = ' '.join(map(str, cmd))
    log(f'$ {shown}')
    process = subprocess.run(cmd, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log(process.stdout.rstrip())
    log(f'[exit {process.returncode}]')
    if process.returncode:
        raise RuntimeError(f'command failed ({process.returncode}): {shown}')
    return process

log('SPATIAL ROOT-PRODUCT LEAN COLAB RUN')
log(f'utc_start={datetime.datetime.now(datetime.timezone.utc).isoformat()}')
log(f'platform={platform.platform()}')
log(f'python={platform.python_version()}')
log(f'cpu_count={os.cpu_count()}')
log(Path('/proc/cpuinfo').read_text(errors='replace').splitlines()[4] if Path('/proc/cpuinfo').exists() else 'cpuinfo=unavailable')
log(Path('/proc/meminfo').read_text(errors='replace').splitlines()[0] if Path('/proc/meminfo').exists() else 'meminfo=unavailable')
log('gpu=none requested or allocated')
run(['git', 'clone', '--filter=blob:none', REPO_URL, str(REPO)])
run(['git', 'checkout', '--detach', EXPECTED_SHA], cwd=REPO)
actual_sha = run(['git', 'rev-parse', 'HEAD'], cwd=REPO).stdout.strip()
if actual_sha != EXPECTED_SHA:
    raise RuntimeError(f'SHA mismatch: {actual_sha} != {EXPECTED_SHA}')
toolchain = (REPO / 'lean-toolchain').read_text(encoding='utf-8').strip()
if toolchain != EXPECTED_TOOLCHAIN:
    raise RuntimeError(f'toolchain mismatch: {toolchain} != {EXPECTED_TOOLCHAIN}')
manifest = json.loads((REPO / 'lake-manifest.json').read_text(encoding='utf-8'))
mathlib_entries = [package for package in manifest['packages'] if package.get('name') == 'mathlib']
if len(mathlib_entries) != 1 or mathlib_entries[0].get('rev') != EXPECTED_MATHLIB:
    raise RuntimeError(f'mathlib pin mismatch: {mathlib_entries}')
gate = REPO / 'scripts' / 'judge_spatial_vacuum_root_products.py'
gate_hash = hashlib.sha256(gate.read_bytes()).hexdigest()
if gate_hash != EXPECTED_GATE_SHA256:
    raise RuntimeError(f'gate hash mismatch: {gate_hash}')
gate_outputs = []
for flags in ([], ['-O']):
    result = run([sys.executable, *flags, str(gate)], cwd=REPO)
    lines = result.stdout.splitlines()
    if len(lines) != 1:
        raise RuntimeError(f'gate emitted stale or extra output under {flags}')
    payload = json.loads(lines[0])
    if payload.get('status') != 'PASS':
        raise RuntimeError(f'gate did not PASS under {flags}: {payload}')
    if payload.get('printed_hypotheses') != ['1 <= L', '0 < x', 'x < 1']:
        raise RuntimeError(f'gate hypotheses changed under {flags}: {payload}')
    if payload.get('mutation_attempts') != payload.get('mutation_rejections'):
        raise RuntimeError(f'gate mutation mismatch under {flags}: {payload}')
    gate_outputs.append(lines[0] + '\n')
gate_output_hash = hashlib.sha256(gate_outputs[0].encode('utf-8')).hexdigest()
if gate_outputs[0] != gate_outputs[1] or gate_output_hash != EXPECTED_GATE_OUTPUT_SHA256:
    raise RuntimeError(f'gate output/hash mismatch: {gate_output_hash}')
log(f'repo_sha={actual_sha}')
log(f'lean_toolchain={toolchain}')
log(f'mathlib_pin={EXPECTED_MATHLIB}')
log(f'gate_sha256={gate_hash}')
log(f'gate_output_sha256={gate_output_hash}')
log('PRECHECK_AND_GATE PASS')

elan_home = RUN_ROOT / 'elan'
env = os.environ.copy()
env['ELAN_HOME'] = str(elan_home)
env['PATH'] = str(elan_home / 'bin') + os.pathsep + env['PATH']
installer = RUN_ROOT / 'elan-init.sh'
run(['curl', '--proto', '=https', '--tlsv1.2', '-sSfL', 'https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh', '-o', str(installer)])
installer_sha = hashlib.sha256(installer.read_bytes()).hexdigest()
run(['sh', str(installer), '-y', '--no-modify-path', '--default-toolchain', 'none'], env=env)
run(['elan', 'toolchain', 'install', EXPECTED_TOOLCHAIN], env=env)
lean_version = run(['lean', '--version'], cwd=REPO, env=env).stdout
if 'version 4.29.0-rc6' not in lean_version or EXPECTED_LEAN_COMMIT not in lean_version:
    raise RuntimeError(f'Lean binary mismatch: {lean_version}')
run(['lake', '--version'], cwd=REPO, env=env)
run(['lake', 'exe', 'cache', 'get'], cwd=REPO, env=env)
log('TOOLCHAIN_AND_CACHE PASS')

module = run(['lake', 'build', 'YangMills.OS.SpatialRing'], cwd=REPO, env=env)
module_match = re.search(r'Build completed successfully \((\d+) jobs\)', module.stdout)
if not module_match:
    raise RuntimeError('module build had no measured job count')
core = run(['lake', 'build', 'YangMillsCore'], cwd=REPO, env=env)
core_match = re.search(r'Build completed successfully \((\d+) jobs\)', core.stdout)
if not core_match or int(core_match.group(1)) != EXPECTED_CORE_JOBS:
    raise RuntimeError(f'core job-count mismatch: {core_match.group(1) if core_match else None}')
oracle_run = run(['lake', 'env', 'lean', 'oracle_check.lean'], cwd=REPO, env=env)
marker = "'YangMills.OS.periodic_antiperiodic_root_products' depends on axioms:"
if marker not in oracle_run.stdout or 'sorryAx' in oracle_run.stdout:
    raise RuntimeError('new declaration missing from clean permanent oracle')
allowed_axioms = {'propext', 'Classical.choice', 'Quot.sound'}
for line in oracle_run.stdout.splitlines():
    if 'depends on axioms:' in line:
        used = {item.strip() for item in line.split('depends on axioms:', 1)[1].strip().strip('[]').split(',') if item.strip()}
        if not used <= allowed_axioms:
            raise RuntimeError(f'nonstandard axioms: {used - allowed_axioms}')
run(['python3', 'scripts/check_consistency.py'], cwd=REPO, env=env)
metadata = {
  'repo_sha': actual_sha, 'toolchain': toolchain, 'lean_commit': EXPECTED_LEAN_COMMIT,
  'mathlib_pin': EXPECTED_MATHLIB, 'gate_sha256': gate_hash,
  'gate_output_sha256': gate_output_hash, 'module_jobs': int(module_match.group(1)),
  'core_jobs': int(core_match.group(1)), 'utc_end': datetime.datetime.now(datetime.timezone.utc).isoformat(),
  'runtime': platform.platform(), 'cpu_count': os.cpu_count(),
  'memory': Path('/proc/meminfo').read_text(errors='replace').splitlines()[0],
  'elan_installer_sha256': installer_sha,
}
(ARTIFACTS / 'metadata.json').write_text(json.dumps(metadata, indent=2, sort_keys=True) + '\n', encoding='utf-8')
(ARTIFACTS / 'oracle_output.txt').write_text(oracle_run.stdout, encoding='utf-8')
hashes = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in sorted(ARTIFACTS.iterdir())}
(ARTIFACTS / 'SHA256SUMS').write_text(''.join(f'{digest}  {name}\n' for name, digest in hashes.items()), encoding='utf-8')
archive = shutil.make_archive('/content/spatial_root_products_lean_artifacts', 'zip', ARTIFACTS)
log(f'module_jobs_measured={module_match.group(1)}')
log(f'core_jobs_measured={core_match.group(1)}')
log(f'artifact_zip={archive}')
log(f'artifact_zip_sha256={hashlib.sha256(Path(archive).read_bytes()).hexdigest()}')
log('SPATIAL ROOT-PRODUCT LEAN PASS')
from google.colab import files
files.download(archive)
